# 01 — Bangun Tabel Qualifying

Notebook ini membaca cache dari `data/raw/` dan menghasilkan dua tabel olahan untuk prediksi kualifikasi:

- `data/quali_lap.csv` — lap kualifikasi yang bersih
- `data/driver_quali.csv` — satu baris per pembalap per qualifying, target = posisi grid

Prasyarat: `00_fetch.ipynb` sudah dijalankan. Semua kode ada di notebook ini,
tidak ada modul Python lokal yang perlu diimpor.

Alur: `data/raw/` → notebook ini → `data/*.csv`.

## Pengenalan Data — Variabel yang Dipakai

### Tabel `driver_quali` (1 baris = 1 pembalap per kualifikasi)

Dibagi menjadi 3 peran: **fitur** (input model), **target** (yang diprediksi), dan **konteks** (identitas baris).

| Kolom | Peran | Sumber | Penjelasan |
|---|---|---|---|
| `meeting_key` | Konteks | `sessions` | ID weekend (satu GP = satu `meeting_key`) |
| `driver_number` | Konteks | `drivers` | Nomor mobil pembalap (mis. 16 = Leclerc) |
| `team_name` | — | `drivers` | **Tidak dipakai** — tim sudah tercermin di fitur pace FP (lap tercepat pembalap) |
| `year` | Konteks | `sessions` | Musim, dipakai untuk split train/test time-based |
| `circuit_short_name` | Konteks | `sessions` | Nama sirkuit (Monza) |
| `fp_best_lap` | (basis) | `laps` (FP1–FP3) | Waktu lap tercepat saat FP — **angka absolut**, bervariasi antar sirkuit; jadi bahan fitur relatif di bawah |
| `fp_median_clean_lap` | (basis) | `laps` (FP1–FP3) | Median lap bersih — pace konsisten (tahan noise lap anomali); bahan fitur relatif |
| `fp_avg_st_speed` | (basis) | `laps` (FP1–FP3) | Rata-rata speed trap — terbukti tidak membedakan pembalap dalam 1 weekend (korelasi −0.050), tidak dipakai model |
| `fp_best_gap` | **Fitur** | turunan `fp_best_lap` | **Selisih detik ke lap tercepat di weekend yang sama** (0 = tercepat) — skala seragam lintas sirkuit |
| `fp_best_rank` | **Fitur** | turunan `fp_best_lap` | Peringkat dalam weekend (1 = tercepat) — persis informasi yang dipakai baseline |
| `fp_median_gap` | **Fitur** | turunan `fp_median_clean_lap` | Selisih pace konsisten (median) ke tercepat di weekend yang sama |
| `hist_grid_season` | **Fitur** | turunan `grid_position` | **Rata-rata grid pembalap di weekend sebelumnya** (musim berjalan) — menangkap skill/mobil yang stabil lintas weekend; dihitung `shift(1)` sehingga hanya memakai masa lalu |
| `hist_n` | **Fitur** | turunan `grid_position` | Jumlah weekend yang sudah dilewati pembalap (pengalaman); kosong = debut |
| `air_temperature` | (konteks) | `weather` (FP) | Suhu udara saat FP — **sama untuk semua pembalap dalam 1 weekend**, tidak dipakai model |
| `track_temperature` | (konteks) | `weather` (FP) | Suhu aspal saat FP — sama untuk semua pembalap dalam 1 weekend, tidak dipakai model |
| `humidity` | (konteks) | `weather` (FP) | Kelembapan — sama untuk semua pembalap dalam 1 weekend, tidak dipakai model |
| `wind_speed` | (konteks) | `weather` (FP) | Kecepatan angin — sama untuk semua pembalap dalam 1 weekend, tidak dipakai model |
| `rainfall` | (konteks) | `weather` (FP) | Indikator hujan saat FP — sama untuk semua pembalap dalam 1 weekend, tidak dipakai model |
| `quali_lap_duration` | (referensi) | `starting_grid` | Waktu lap terbaik di kualifikasi — hanya referensi, **bukan fitur** (leakage) |
| `grid_position` | **TARGET** | `starting_grid` | Posisi grid hasil kualifikasi — inilah yang diprediksi model |
| `is_top3` | **TARGET** turunan | `starting_grid` | 1 jika `grid_position ≤ 3` (pole + baris depan) |
| `is_top10` | **TARGET** turunan | `starting_grid` | 1 jika `grid_position ≤ 10` (Q3) |

### Tabel `quali_lap` (1 baris = 1 lap kualifikasi)

| Kolom | Sumber | Penjelasan |
|---|---|---|
| `session_key`, `meeting_key` | `laps` | ID sesi dan weekend |
| `driver_number` | `laps` | Pembalap yang melaju |
| `lap_number` | `laps` | Nomor urut lap dalam sesi |
| `lap_duration` | `laps` | Waktu lap (detik) — bahan analisis pacing kualifikasi |
| `duration_sector_1/2/3` | `laps` | Waktu per sektor — untuk tahu kekuatan di sektor mana |
| `i1_speed`, `i2_speed`, `st_speed` | `laps` | Kecepatan di tiap checkpoint & speed trap |
| `is_pit_out_lap` | `laps` | Lap keluar pit (biasanya lebih lambat, sering dibuang) |

### Kenapa variabel lain tidak dipakai?

| Data | Alasan dikecualikan |
|---|---|
| Lap Qualifying sebagai **fitur** | **Leakage** — hasil kualifikasi tidak boleh dipakai untuk memprediksi kualifikasi |
| Hasil race (`position_finish`, `points`, `duration`) | Terjadi **setelah** qualifying — di luar cut-off |
| `starting_grid.lap_duration` sebagai fitur | Bagian dari hasil qualifying → leakage. Hanya dipakai sebagai referensi/analisis |
| `championship_*` | Klasemen dihitung dari hasil race → mengandung info post-quali |

**Analogi:** untuk menebak urutan siswa masuk kelas sebelum ujian, kamu boleh pakai nilai ulangan kemarin (FP), tapi tidak boleh pakai nilai uciannya sendiri (hasil Qualifying).

## 0. Setup & loader

Cell ini:

- membaca `config.yaml` dan menentukan root project,
- membuat `load_endpoint(nama)` untuk menggabungkan semua file cache satu endpoint (`.parquet` dan `.json`),
- memberi pesan jelas kalau cache belum ada.

Output: lokasi folder `data/raw/` yang dipakai.

In [1]:
import pandas as pd
import yaml
from pathlib import Path

# Cari root project. Aman dijalankan dari folder notebooks/ maupun dari root.
ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists() and (ROOT.parent / "config.yaml").exists():
    ROOT = ROOT.parent

with (ROOT / "config.yaml").open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle) or {}


def resolve_path(value):
    path = Path(value)
    return path if path.is_absolute() else ROOT / path


raw = resolve_path(cfg["paths"]["raw"])


def load_endpoint(name):
    folder = raw / name
    files = sorted(folder.glob("*.parquet")) + sorted(folder.glob("*.json"))
    if not files:
        raise FileNotFoundError(
            f"Belum ada cache untuk '{name}'. Jalankan 00_fetch.ipynb dulu."
        )
    frames = [
        pd.read_parquet(f) if f.suffix == ".parquet" else pd.read_json(f)
        for f in files
    ]
    return pd.concat(frames, ignore_index=True)


print("Raw dir:", raw)

Raw dir: C:\syntax\codelab\SubrisetData\prediksi-openf1\data\raw


## 1. Tentukan sesi Qualifying

`starting_grid` tersedia di sesi Qualifying. Kita ambil daftar `session_key` Qualifying
dari cache `starting_grid` dan `sessions`.

Output: jumlah qualifying session yang ditemukan di cache.

In [2]:
starting_grid = load_endpoint("starting_grid")
sessions = load_endpoint("sessions")

quali_keys = set(starting_grid["session_key"].unique())
print("Jumlah qualifying session:", len(quali_keys))

Jumlah qualifying session: 90


## 2. Bangun `quali_lap`

Satu baris = satu lap dari sesi Qualifying. Lap yang dipertahankan:

- `lap_duration` tidak kosong,
- bukan pit-out lap.

Output: tabel `quali_lap` dan 5 baris pertama.

In [3]:
laps = load_endpoint("laps")
laps = laps[laps["session_key"].isin(quali_keys)].copy()

quali_lap = laps[
    laps["lap_duration"].notna()
    & (~laps["is_pit_out_lap"].fillna(False))
].copy()

quali_lap = quali_lap.reset_index(drop=True)

print("Lap qualifying:", quali_lap.shape)
quali_lap.head()

Lap qualifying: (19346, 16)


C:\Users\Rizkya Gusnaldy\AppData\Local\Temp\ipykernel_27620\2194019549.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


,meeting_key,session_key,driver_number,lap_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,is_pit_out_lap,lap_duration,segments_sector_1,segments_sector_2,segments_sector_3,st_speed
0,1256,10002,31,2,2025-04-05T06:01:54.928000+00:00,31.371,40.473,17.880,284.0,296.0,False,89.724,"[2049.0, 2049.0, 2049.0, 2049.0, 2049.0, 2049....","[2049, 2049, 2049, 2051, 2049, 2049, 2049, 204...","[2049, 2049, 2051, 2049, 2051]",291.0
1,1256,10002,87,2,2025-04-05T06:02:02.428000+00:00,31.363,40.350,17.721,285.0,296.0,False,89.434,"[2049.0, 2049.0, 2049.0, 2049.0, 2049.0, 2049....","[2049, 2049, 2051, 2049, 2051, 2049, 2049, 204...","[2049, 2049, 2051, 2051, 2051, 2048]",291.0
2,1256,10002,7,2,2025-04-05T06:02:10.990000+00:00,31.482,40.583,17.738,285.0,298.0,False,89.803,"[2049.0, 2051.0, 2049.0, 2049.0, 2049.0, 2049....","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2048, 2049]",295.0
3,1256,10002,55,2,2025-04-05T06:02:18.568000+00:00,31.451,40.390,17.682,284.0,302.0,False,89.523,"[2049.0, 2049.0, 2049.0, 2049.0, 2049.0, 2049....","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 205...","[2051, 2051, 2049, 2048, 2049, 2048]",295.0
4,1256,10002,27,2,2025-04-05T06:02:28.131000+00:00,31.460,40.333,17.719,285.0,296.0,False,89.512,"[2049.0, 2049.0, 2049.0, 2049.0, 2049.0, 2049....","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2051, 2049, 2049]",291.0


## 3. Bangun `driver_quali`

Satu baris = satu pembalap dalam satu qualifying session. Fiturnya berasal dari
FP1–FP3 (sebelum cut-off), target-nya adalah `grid_position` dari `starting_grid`.

Gabungan:

- fitur pace FP (best lap, median clean lap, avg speed trap) per pembalap,
- cuaca FP,
- grid position dari `starting_grid` (target),
- metadata `year` dan `circuit_short_name` dari `sessions`.

Kolom turunan:

- `grid_position` — posisi start dari kualifikasi (**target**),
- `is_top3` — 1 kalau grid ≤ 3,
- `is_top10` — 1 kalau grid ≤ 10.

In [4]:
# Ambil sesi FP (Practice 1/2/3) untuk fitur
fp_sessions = sessions[
    sessions["session_name"].isin(["Practice 1", "Practice 2", "Practice 3"])
].copy()
fp_keys = set(fp_sessions["session_key"].unique())

# Laps FP per pembalap
fp_laps = load_endpoint("laps")
fp_laps = fp_laps[fp_laps["session_key"].isin(fp_keys)].copy()
fp_clean = fp_laps[
    fp_laps["lap_duration"].notna()
    & (fp_laps["lap_number"] > 1)
    & (~fp_laps["is_pit_out_lap"].fillna(False))
].copy()

# Agregasi FP per pembalap per meeting
fp_agg = (
    fp_clean.groupby(["meeting_key", "driver_number"])
    .agg(
        fp_best_lap=("lap_duration", "min"),
        fp_median_clean_lap=("lap_duration", "median"),
        fp_avg_st_speed=("st_speed", "mean"),
    )
    .reset_index()
)

# ---- Fitur RELATIF per weekend ----
# Kenapa: target grid_position itu RELATIF dalam 1 weekend (urutan dalam GP),
# sedangkan fp_best_lap absolut bervariasi antar sirkuit (Monza 79s, Spa 106s).
# Model yang belajar angka absolut justru belajar mengenali sirkuit, bukan pembalap.
# Bukti: korelasi fp_best_lap->grid POOL = 0.061 (nyaris nol), PER-WEEKEND = 0.657 (kuat).
# Dengan fitur gap/rank, model bicara bahasa yang sama dengan target.
fp_agg["fp_best_gap"] = fp_agg.groupby("meeting_key")["fp_best_lap"].transform(
    lambda s: s - s.min()
)   # selisih detik ke lap tercepat weekend (0 = tercepat)
fp_agg["fp_best_rank"] = fp_agg.groupby("meeting_key")["fp_best_lap"].rank(
    method="min"
)   # peringkat dalam weekend (1 = tercepat) = informasi yang dipakai baseline
fp_agg["fp_median_gap"] = fp_agg.groupby("meeting_key")["fp_median_clean_lap"].transform(
    lambda s: s - s.min()
)   # selisih pace konsisten (median) ke tercepat

# Cuaca FP (rata-rata per meeting)
weather = load_endpoint("weather")
weather = weather[weather["session_key"].isin(fp_keys)].copy()
weather_cols = [
    "session_key", "air_temperature", "track_temperature",
    "humidity", "wind_speed", "rainfall"
]
weather_agg = (
    weather[weather_cols]
    .groupby("session_key")[[
        "air_temperature", "track_temperature",
        "humidity", "wind_speed", "rainfall"
    ]]
    .mean()
    .reset_index()
)

# Map weather ke meeting via sessions
sess_map = sessions[["session_key", "meeting_key"]].drop_duplicates()
weather_agg = weather_agg.merge(sess_map, on="session_key", how="left")
weather_meeting = (
    weather_agg.groupby("meeting_key")[[
        "air_temperature", "track_temperature",
        "humidity", "wind_speed", "rainfall"
    ]]
    .mean()
    .reset_index()
)

# Gabung FP fitur + cuaca per meeting
driver_quali = fp_agg.merge(weather_meeting, on="meeting_key", how="left")

# Gabung grid position (target) dari starting_grid
grid = starting_grid[["meeting_key", "driver_number", "position", "lap_duration"]]
grid = grid.rename(columns={"position": "grid_position", "lap_duration": "quali_lap_duration"})

driver_quali = driver_quali.merge(grid, on=["meeting_key", "driver_number"], how="left")

# Metadata year & circuit
sess_meta = sessions[["meeting_key", "year", "circuit_short_name"]].drop_duplicates()
driver_quali = driver_quali.merge(sess_meta, on="meeting_key", how="left")

# Target turunan
driver_quali["is_top3"] = (driver_quali["grid_position"] <= 3).astype("Int64")
driver_quali["is_top10"] = (driver_quali["grid_position"] <= 10).astype("Int64")

# ---- Fitur HISTORI (hanya data MASA LALU — anti leakage) ----
# Kenapa: fitur dalam-weekend (gap/rank) sudah maksimal tapi belum mengalahkan
# baseline. Yang kurang adalah 'skill pembalap' yang tidak terlihat dari 1 weekend FP.
# Rata-rata grid pembalap di weekend SEBELUMNYA menangkap hal itu.
# Kunci anti-leakage: sort chronologis + shift(1)/expanding().mean().shift(1)
# → tiap baris HANYA memakai grid dari weekend yang sudah lewat.
driver_quali = driver_quali.sort_values(
    ["year", "meeting_key", "driver_number"]
).reset_index(drop=True)

by_driver = driver_quali.groupby("driver_number")["grid_position"]
driver_quali["hist_grid_season"] = by_driver.transform(
    lambda s: s.expanding().mean().shift(1)
)   # rata-rata grid pembalap sejauh musim ini, TANPA weekend berjalan
driver_quali["hist_n"] = by_driver.transform(
    lambda s: s.expanding().count().shift(1)
)   # jumlah weekend terlewati (pengalaman; NaN = debut, biarkan kosong)

print("Driver-quali:", driver_quali.shape)
print("Baris tanpa grid (target kosong):", driver_quali["grid_position"].isna().sum())
driver_quali.head()

C:\Users\Rizkya Gusnaldy\AppData\Local\Temp\ipykernel_27620\2194019549.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


Driver-quali: (1974, 21)
Baris tanpa grid (target kosong): 140


,meeting_key,driver_number,fp_best_lap,fp_median_clean_lap,fp_avg_st_speed,fp_best_gap,fp_best_rank,fp_median_gap,air_temperature,track_temperature,...,wind_speed,rainfall,grid_position,quali_lap_duration,year,circuit_short_name,is_top3,is_top10,hist_grid_season,hist_n
0,1141,1,91.076,98.5675,265.173913,0.169,2.0,0.4210,25.954076,36.008671,...,1.813341,0.0,1.0,89.708,2023,Sakhir,1,1,NaN,NaN
1,1141,2,92.749,99.9630,292.673077,1.842,20.0,1.8165,25.954076,36.008671,...,1.813341,0.0,16.0,91.652,2023,Sakhir,0,0,NaN,NaN
2,1141,4,91.570,99.7190,276.440000,0.663,9.0,1.5725,25.954076,36.008671,...,1.813341,0.0,11.0,91.381,2023,Sakhir,0,0,NaN,NaN
3,1141,10,91.475,100.0480,290.640000,0.568,7.0,1.9015,25.954076,36.008671,...,1.813341,0.0,20.0,92.181,2023,Sakhir,0,0,NaN,NaN
4,1141,11,91.078,98.2030,272.590909,0.171,3.0,0.0565,25.954076,36.008671,...,1.813341,0.0,2.0,89.846,2023,Sakhir,1,1,NaN,NaN


## 4. Simpan tabel

Kedua tabel ditulis ke path dari `config.yaml`:

- `paths.quali_lap` → `data/quali_lap.csv`
- `paths.driver_quali` → `data/driver_quali.csv`

Output: dua baris `Saved: <path>`.

In [5]:
quali_lap_path = resolve_path(cfg["paths"]["quali_lap"])
driver_quali_path = resolve_path(cfg["paths"]["driver_quali"])

quali_lap.to_csv(quali_lap_path, index=False)
driver_quali.to_csv(driver_quali_path, index=False)

print("Saved:", quali_lap_path)
print("Saved:", driver_quali_path)

Saved: C:\syntax\codelab\SubrisetData\prediksi-openf1\data\quali_lap.csv
Saved: C:\syntax\codelab\SubrisetData\prediksi-openf1\data\driver_quali.csv


## 5. Cek cepat

Yang perlu dilihat:

- `quali_lap` dan `driver_quali` tidak kosong,
- jumlah qualifying session sesuai yang di-fetch,
- jumlah baris tanpa grid (target kosong) = 0,
- distribusi `grid_position` adalah 1–20.

In [6]:
print("quali_lap:", quali_lap.shape)
print("driver_quali:", driver_quali.shape)
print("Qualifying unik:", driver_quali["meeting_key"].nunique())
print("Baris tanpa grid:", driver_quali["grid_position"].isna().sum())
print("Distribusi grid_position:")
print(driver_quali["grid_position"].value_counts().sort_index().head(20))

quali_lap: (19346, 16)
driver_quali: (1974, 21)
Qualifying unik: 91
Baris tanpa grid: 140
Distribusi grid_position:
grid_position
1.0     90
2.0     90
3.0     90
4.0     90
5.0     90
6.0     89
7.0     90
8.0     90
9.0     90
10.0    89
11.0    90
12.0    90
13.0    90
14.0    90
15.0    90
16.0    90
17.0    90
18.0    90
19.0    89
20.0    87
Name: count, dtype: int64


## Ringkasan output `01_tables.ipynb`

| File | Isi | Dipakai untuk |
|---|---|---|
| `data/quali_lap.csv` | 1 baris per lap qualifying yang bersih | Analisis pacing kualifikasi |
| `data/driver_quali.csv` | 1 baris per pembalap per qualifying | Model prediksi posisi grid |

Belum ada file di `outputs/`. Model dan metrik menyusul setelah kedua tabel ini diverifikasi.